In [2]:
import boto3
import json
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def get_embedding(prompt_data, boto3_bedrock):
    # Titan v1 acepta máximo 50,000 caracteres — truncamos para evitar errores
    prompt_data = prompt_data[:25000]  # ~6,500 tokens — bajo el límite de 8,192 de Titan v1
    body = json.dumps({"inputText": prompt_data})
    modelId = "amazon.titan-embed-text-v1"  # (Change this to try different embedding models)
    accept = "application/json"
    contentType = "application/json"

    response = boto3_bedrock.invoke_model(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    response_body = json.loads(response.get("body").read())

    embedding = response_body.get("embedding")
    return embedding


def invoke_model(prompt_data, question, boto3_bedrock):

    model_input_dict = {'anthropic_version': 'bedrock-2023-05-31',
                        'max_tokens': 20000,
                        'system': prompt_data,
                        "messages" : question}

    body = json.dumps(model_input_dict)

    modelId = 'us.anthropic.claude-sonnet-4-6'  # change this to use a different version from the model provider
    accept = "application/json"
    contentType = "application/json"

    response = boto3_bedrock.invoke_model(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    response_body = json.loads(response.get("body").read())

    response_string = response_body['content'][0]['text']
    response_string = response_string.strip()

    return response_string


def answer_question(question, boto3_bedrock, prompt=""):
    user_request = "<request>" + question + "</request>"
    messages = [{"role": "user", "content": user_request}]
    return invoke_model(prompt, messages, boto3_bedrock)


def compute_closest_texts(question, embeddings, boto3_bedrock, embedding_column = 'embedding'):
    target_embedding = get_embedding(question, boto3_bedrock)

    vector_array = np.array(target_embedding)
    matrix_array = np.array([emb for emb in embeddings[embedding_column]])

    vector_array = vector_array.reshape(1, -1)

    # Compute cosine similarity
    similarities = cosine_similarity(vector_array, matrix_array)

    n_texts = len(similarities)-1
    
    # Flatten the similarities matrix and get the indices of the top 3 similarities
    top_indices = np.argpartition(similarities.flatten(), -n_texts)[-n_texts:]
    top_indices = top_indices[np.argsort(similarities.flatten()[top_indices])][::-1]
    
    return top_indices


def enhance_prompt(question, df, boto3_bedrock, num_contexts=5):
    top_indices=compute_closest_texts(question, df, boto3_bedrock, embedding_column = 'embedding')

    prompt = f"""
    <task_instructions> Eres un experto en política colombiana. El usuario te va a hacer una pregunta
    y tú debes responderla basado en los textos de contexto que están a continuación. Evita
    decir cosas que no se mencionen en los textos. Si los textos no contienen información relevante para
    responder la pregunta, responde que no tienes una respuesta. En tu respuesta evita mencionar los textos,
    con frases como "de acuerdo con los textos proporcionados". Tu respuesta debe ser escrita en un párrafo y evita
    escribirla como una lista de puntos.
    """

    for i in range(num_contexts):
        prompt += f""" 
        <context_text>
        {df.loc[top_indices[i], 'content']}
        </context_text>
        """

    return prompt

Inicializar el cliente a Bedrock:

In [3]:
boto3_bedrock = boto3.client(
    service_name='bedrock-runtime',
    region_name="us-east-1"
)

In [4]:
model_input_dict = {'anthropic_version': 'bedrock-2023-05-31',
                    'max_tokens': 20000,
                    'system': '',
                    "messages" : [{"role": "user", "content": "Cuál es la capital de Colombia?"}]}

body = json.dumps(model_input_dict)

modelId = 'us.anthropic.claude-sonnet-4-6'  # change this to use a different version from the model provider
accept = "application/json"
contentType = "application/json"

response = boto3_bedrock.invoke_model(
    body=body, modelId=modelId, accept=accept, contentType=contentType
)
response_body = json.loads(response.get("body").read())

response_string = response_body['content'][0]['text']
response_string = response_string.strip()

print(response_string)

La capital de Colombia es **Bogotá**, cuyo nombre oficial es **Bogotá, Distrito Capital (D.C.)**. Es la ciudad más grande del país y está ubicada en el centro de Colombia, en la sabana del altiplano cundiboyacense, a aproximadamente 2.600 metros sobre el nivel del mar.


### Ejemplo sin prompt:

In [5]:
question='Cuéntame de la reforma a la salud'

print(answer_question(question, boto3_bedrock))

# Reforma a la Salud

Para darte información precisa, necesito saber **¿de qué país estás hablando?**, ya que hay varias reformas de salud relevantes actualmente.

Sin embargo, si me preguntas por la más comentada recientemente, probablemente sea la de **Colombia**:

---

## 🇨🇴 Reforma a la Salud en Colombia

### ¿Qué propone?
- **Eliminar gradualmente las EPS** como intermediarias
- Crear un sistema más **público y descentralizado**
- Fortalecer la **atención primaria** y los centros de salud locales
- El Estado manejaría directamente los recursos del sistema

### Puntos a favor (según sus defensores)
- Mayor cobertura para zonas rurales
- Eliminar la corrupción en intermediarios
- Más recursos llegando directamente a pacientes

### Críticas y controversias
- Riesgo de **colapso del sistema** por cambio abrupto
- Dudas sobre la **capacidad institucional** del Estado
- Oposición de médicos, clínicas y parte del Congreso
- Debate sobre su **viabilidad financiera**

### Estado actual
Ha 

### Retrieval Augmented Generation (RAG):

In [6]:
df = pd.read_csv("../datasets/noticias_politica_sample.csv").dropna(subset=["content"]).reset_index(drop=True)
print(f"Noticias cargadas: {len(df)}")

Noticias cargadas: 495


In [7]:
embeddings = []
for i in range(len(df)):
    embeddings.append(get_embedding(df.loc[i, 'content'], boto3_bedrock))

df['embedding'] = embeddings

In [8]:
df

,content,date,headline,description,embedding
0,"Esta semana, el Senado y la Cámara de Represen...",2022-11-10T18:42:14.884Z,"“Con algunas excepciones, un abusivo es un pas...",El senador del Pacto Histórico se refirió al c...,"[0.5341837406158447, -0.18686246871948242, 0.4..."
1,"Hace pocas horas, el expresidente interino de ...",2023-04-24T13:58:31.866Z,Gustavo Bolívar se pronunció sobre visita a Co...,El expresidente interino de Venezuela llegará ...,"[0.3212205171585083, -0.1835937649011612, 0.26..."
2,En medio de la preocupación que han elevado va...,2023-03-24T14:53:30.902Z,“Recibimos el país inundado en coca”: fuerte s...,El funcionario diplomático del Gobierno del pr...,"[0.6057479381561279, -0.35421520471572876, 0.2..."
3,"Este lunes, de cara a una semana clave para la...",2023-04-24T20:03:25.019Z,“Esperamos que entre miércoles y jueves se ten...,El presidente de la Cámara de Representantes e...,"[0.6127364039421082, -0.04097236692905426, 0.3..."
4,"Sin duda, el presidente Gustavo Petro, el mini...",2023-03-24T02:57:29.758Z,"Tras fracaso de la reforma política, Gobierno ...",Aunque esa decisión se pudo tomar desde el sem...,"[0.4289451241493225, -0.11647289246320724, 0.4..."
...,...,...,...,...,...
490,Más de 50 representantes del sector emprendedo...,2022-10-19T00:18:36.429Z,“Nos toca irnos del país para seguir subsistie...,Los voceros de pequeñas y medianas empresas as...,"[0.1192505732178688, -0.10446195304393768, 0.2..."
491,"Gilberto Tobón Sanín, excandidato al Senado de...",2022-11-30T05:02:30.124Z,"“No le hago mandados a nadie, soy autónomo”: G...",El aspirante afirmó que su “compromiso es con ...,"[0.30228325724601746, -0.2461211383342743, 0.3..."
492,Las diferencias entre el senador de la bancada...,2022-11-30T15:18:07.075Z,Gustavo Bolívar arremete en contra del senador...,"El ministro de Hacienda, presente en el debate...","[0.37524884939193726, -0.1569998413324356, 0.1..."
493,"En la tarde de este jueves, 24 de noviembre, e...",2022-11-25T04:26:26.611Z,"El ministro del Interior, Alfonso Prada, fue d...",Con el Puerto Pisisí esperan mover 1. 630.000 ...,"[0.5068084001541138, -0.23955419659614563, 0.0..."


In [9]:
df['embedding']

0      [0.5341837406158447, -0.18686246871948242, 0.4...
1      [0.3212205171585083, -0.1835937649011612, 0.26...
2      [0.6057479381561279, -0.35421520471572876, 0.2...
3      [0.6127364039421082, -0.04097236692905426, 0.3...
4      [0.4289451241493225, -0.11647289246320724, 0.4...
                             ...                        
490    [0.1192505732178688, -0.10446195304393768, 0.2...
491    [0.30228325724601746, -0.2461211383342743, 0.3...
492    [0.37524884939193726, -0.1569998413324356, 0.1...
493    [0.5068084001541138, -0.23955419659614563, 0.0...
494    [0.809524416923523, 0.08103227615356445, 0.410...
Name: embedding, Length: 495, dtype: object

In [10]:
df.loc[0, 'content']

'Esta semana, el Senado y la Cámara de Representantes se ponen de acuerdo para conciliar la reforma tributaria que aprobó el legislativo. Uno de los puntos de la discordia es el artículo que le impone un 20 % de impuestos a las iglesias en Colombia, una iniciativa que respaldó la Cámara, pero rechazó el grueso de senadores. El tiempo apremia y la idea es que el presidente Gustavo Petro sancione el proyecto conciliado este jueves 10 de noviembre. Como el punto de diferencia entre el Senado y la Cámara es la tributación a las iglesias y a los negocios que se deriven de ella, el senador del Pacto Histórico, Gustavo Bolívar, defendió este jueves el artículo. “Con algunas excepciones, un abusivo es un pastor que se enriquece infundiendo temor a Dios. La manipulación de la fe con fines económicos debería ser un delito”, dijo. Y posteriormente preguntó: “¿Por qué no pagan impuestos si son entidades sin ánimo de lucro?”. Bolívar es consciente que el escenario político en la conciliación para a

In [11]:
question='Cuéntame de la reforma a la salud'

In [12]:
top_indices=compute_closest_texts(question, df, boto3_bedrock, embedding_column = 'embedding')

/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [13]:
df.loc[top_indices[0], 'content']

'Luego de que se llevó a cabo un consejo de ministros que tuvo lugar en Villa de Leyva, Boyacá, el Gobierno Nacional dio a conocer que la reforma a la salud iniciará debate a partir del próximo 6 de febrero, lo que quiere decir que se presentará en las sesiones extraordinarias del Congreso. “La discusión y análisis de la reforma se centró en la garantía necesaria de la salud como un derecho fundamental, el incremento del acceso, los modelos de transición hacia un nuevo sistema y la adopción de medidas para la mejora de indicadores de vida, a través de la atención primaria”, señaló la Presidencia de la República mediante un comunicado. En la reunión de ministros, el presidente Gustavo Petro pidió priorizar las grandes reformas y los asuntos programáticos que sean transversales a la gestión de los ministerios. Entre esas prioridades se encuentra la reforma a la salud, aunque cabe mencionar que las críticas sobre esta decisión no han cesado, puesto que la vida e integridad de la ciudadaní

In [14]:
df.loc[top_indices[1], 'content']

'Este martes 24 de enero, el Pacto Histórico inició sus retiros espirituales en los que el Gobierno presentó oficialmente a las bancadas de Senado y Cámara del petrismo las reformas que llegarán al legislativo en las próximas semanas. Aunque la ministra de Salud, Carolina Corcho, estuvo presente e intervino durante una hora, no socializó oficialmente el contenido final del proyecto. Era el documento más esperado de la jornada, pero no fue entregado a las bancadas. Así lo manifestaron a SEMANA varios congresistas de la izquierda, y además así está probado en la intervención de la funcionaria que tiene en su poder esta revista. Lo que hizo fue socializar algunas pinceladas, pero no abordó la posible eliminación de las Empresas Prestadoras de Salud (EPS), el contenido más sensible del proyecto de ley. De entrada, Corcho aseguró que la reforma a la salud ha tenido dos debates de control político sin ser radicada en el Congreso. “No conozco en la historia del país que un ministro haya sido 

In [15]:
df.loc[top_indices[2], 'content']

'El presidente de la República, Gustavo Petro, sorpresivamente este lunes 27 de febrero reconoció que la aguda carta que le enviaron varios de sus ministros criticando la polémica reforma a la salud es real y reveló los cambios que el documento promovió a la iniciativa. De acuerdo con el mandatario colombiano, la misiva que envió el ministro de Educación, Alejandro Gaviria (abierto contradictor de la reforma); la ministra de Agricultura, Cecilia López, y el ministro de Hacienda, José Antonio Ocampo, no pasó desapercibida; al contrario, generó que el presidente Petro convocara a una reunión extraordinaria a los representantes de nueve EPS, reunión que se llevó a cabo a pocas horas de ser radicada la reforma a la salud en el Congreso de la República. “La carta publicada por medios de comunicación por algunos de mis ministros sobre la reforma a la salud, es cierta. Fue discutida por horas y por días, por mí mismo y por los equipos técnicos que se designaron para ello. Hice reunión con las

In [17]:
top_indices=compute_closest_texts(question, df, boto3_bedrock, embedding_column = 'embedding')

prompt = f"""
<task_instructions> Eres un experto en política colombiana. El usuario te va a hacer una pregunta
y tú debes responderla basado en los textos de contexto que están a continuación. Evita
decir cosas que no se mencionen en los textos. Tu respuesta debe ser escrita en un párrafo y evita
escribirla como una lista de puntos.
"""

num_contexts = 5

for i in range(num_contexts):
    prompt += f""" 
    <context_text>
    {df.loc[top_indices[i], 'content']}
    </context_text>
    """

/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [18]:
print(prompt)


<task_instructions> Eres un experto en política colombiana. El usuario te va a hacer una pregunta
y tú debes responderla basado en los textos de contexto que están a continuación. Evita
decir cosas que no se mencionen en los textos. Tu respuesta debe ser escrita en un párrafo y evita
escribirla como una lista de puntos.
 
    <context_text>
    Luego de que se llevó a cabo un consejo de ministros que tuvo lugar en Villa de Leyva, Boyacá, el Gobierno Nacional dio a conocer que la reforma a la salud iniciará debate a partir del próximo 6 de febrero, lo que quiere decir que se presentará en las sesiones extraordinarias del Congreso. “La discusión y análisis de la reforma se centró en la garantía necesaria de la salud como un derecho fundamental, el incremento del acceso, los modelos de transición hacia un nuevo sistema y la adopción de medidas para la mejora de indicadores de vida, a través de la atención primaria”, señaló la Presidencia de la República mediante un comunicado. En la reun

In [19]:
print(answer_question(question, boto3_bedrock, prompt))

La reforma a la salud en Colombia, liderada por la ministra Carolina Corcho, fue presentada por el Gobierno del presidente Gustavo Petro como una de sus principales prioridades, con el objetivo de garantizar la salud como derecho fundamental, ampliar el acceso, mejorar la atención primaria y llevar los servicios médicos a los territorios más apartados del país. La reforma propone eliminar la intermediación financiera de las EPS en el manejo de los recursos públicos, trasladando los pagos de manera directa a clínicas, hospitales y centros de salud, lo que según el Gobierno generaría un ahorro de $6,5 billones anuales al sistema; sin embargo, este ha sido el punto más polémico, pues los directivos de las EPS han señalado que se busca acabar con ellas, mientras la ministra insiste en que tendrán una función alternativa dentro del nuevo modelo. El texto de la reforma, que inicialmente contaba con 156 artículos y fue ajustado en aproximadamente un 40%, también contempla cambios en la políti

In [20]:
question='Qué ha pasado con Nicolás Petro?'
prompt = enhance_prompt(question, df, boto3_bedrock)

print(answer_question(question, boto3_bedrock, prompt))

/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Nicolás Petro, hijo mayor del presidente Gustavo Petro y diputado de la Asamblea del Atlántico, se ha visto envuelto en un grave escándalo tras las declaraciones de su exesposa Days Vásquez en una entrevista exclusiva con la revista Semana, en la que aseguró que él recibió más de 600 millones de pesos del exnarcotraficante Santander Lopesierra (el Hombre Marlboro) y de Alfonso "El Turco Hilsaca" para la campaña presidencial de su padre, dineros que supuestamente nunca ingresaron legalmente a la campaña porque Nicolás se los habría apropiado. Además, se reveló a través de chats que en su apartamento, un penthouse valorado en 2.500 millones de pesos en Barranquilla, se manejaban grandes sumas de dinero en efectivo que eran trasladadas de forma clandestina entre varias personas de confianza para evitar sospechas. A esto se suma una segunda acusación relacionada con el supuesto cobro de dinero a narcos a cambio de incluirlos en la política de 'paz total' del Gobierno. Ante el escándalo, el